In [1]:
# ============================================================================
# CELL 1: Import và Setup
# ============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.models as models
from torchvision import transforms, datasets
import math
import numpy as np

print("Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


Imports successful!
PyTorch version: 2.6.0+cu124
CUDA available: True


In [2]:
# ============================================================================
# CELL 2: APBLayer Class (Fixed inference_forward)
# ============================================================================
class APBLayer(nn.Module):
    def __init__(self, layer_to_wrap: nn.Module):
        super().__init__()
        if not isinstance(layer_to_wrap, (nn.Linear, nn.Conv2d)):
            raise ValueError("APBLayer chỉ hỗ trợ nn.Linear và nn.Conv2d.")

        self.wrapped_layer = layer_to_wrap
        self.latent_weight = nn.Parameter(layer_to_wrap.weight.data.clone())
        self.bias = layer_to_wrap.bias
        if hasattr(self.wrapped_layer, 'weight'):
            del self.wrapped_layer.weight

        with torch.no_grad():
            weights = self.latent_weight.data
            initial_alpha = weights.abs().mean()
            initial_delta = 3.0 * weights.std().clamp(min=1e-5)
        self.alpha = nn.Parameter(torch.tensor(initial_alpha.item(), device=weights.device))
        self.delta = nn.Parameter(torch.tensor(initial_delta.item(), device=weights.device))

    def forward(self, x):
        delta_clamped = self.delta.clamp(min=1e-8)
        w_hat = (self.latent_weight.abs() - self.alpha.abs()) / delta_clamped
        binarization_mask = (w_hat <= 1.0)
        sign_tensor = torch.sign(self.latent_weight)
        binarized_part = binarization_mask * sign_tensor * self.alpha.abs()
        full_precision_part = ~binarization_mask * self.latent_weight
        effective_weight = binarized_part + full_precision_part
        effective_weight = self.latent_weight + (effective_weight - self.latent_weight).detach()

        if isinstance(self.wrapped_layer, nn.Linear):
            return F.linear(x, effective_weight, self.bias)
        elif isinstance(self.wrapped_layer, nn.Conv2d):
            return F.conv2d(
                x, effective_weight, self.bias,
                self.wrapped_layer.stride, self.wrapped_layer.padding,
                self.wrapped_layer.dilation, self.wrapped_layer.groups
            )

    def get_stats(self):
        with torch.no_grad():
            total_weights = self.latent_weight.numel()
            threshold = self.alpha.abs() + self.delta.clamp(min=1e-8)
            num_binary = (self.latent_weight.abs() <= threshold).sum().item()
            return {
                "alpha": self.alpha.item(),
                "delta": self.delta.item(),
                "percent_binary": (num_binary / total_weights) * 100,
            }

    def get_effective_weight(self):
        delta_clamped = self.delta.clamp(min=1e-8)
        threshold = self.alpha.abs() + delta_clamped
        binarization_mask = self.latent_weight.abs() <= threshold
        sign_tensor = torch.ones_like(self.latent_weight)
        sign_tensor[self.latent_weight < 0] = -1
        binarized_part = torch.where(binarization_mask, 
                                   sign_tensor * self.alpha,
                                   torch.zeros_like(self.latent_weight))
        full_precision_part = torch.where(~binarization_mask, 
                                        self.latent_weight, 
                                        torch.zeros_like(self.latent_weight))
        return binarized_part + full_precision_part

print("APBLayer class defined!")

APBLayer class defined!


In [3]:
# ============================================================================
# CELL 3: Apply APB Function
# ============================================================================
def apply_apb(model: nn.Module, skip_first_conv=True, skip_downsample=True, skip_last_linear=True):
    conv_layers = [(name, module) for name, module in model.named_modules() if isinstance(module, nn.Conv2d)]
    first_conv_name = conv_layers[0][0] if conv_layers and skip_first_conv else None
    linear_layers = [(name, module) for name, module in model.named_modules() if isinstance(module, nn.Linear)]
    last_linear_name = linear_layers[-1][0] if linear_layers and skip_last_linear else None

    for name, module in list(model.named_modules()):
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            is_first_conv = (name == first_conv_name) and isinstance(module, nn.Conv2d)
            is_downsampling = isinstance(module, nn.Conv2d) and (any(s > 1 for s in module.stride) or 'downsample' in name)
            is_last_linear = (name == last_linear_name) and isinstance(module, nn.Linear)

            if is_first_conv or is_downsampling or is_last_linear:
                print(f"Skipping APB for: {name}")
                continue

            print(f"Applying APB to: {name}")
            parent_name = '.'.join(name.split('.')[:-1])
            child_name = name.split('.')[-1]
            parent_module = model
            if parent_name:
                for part in parent_name.split('.'):
                    parent_module = getattr(parent_module, part)
            setattr(parent_module, child_name, APBLayer(module))

    return model

print("apply_apb function defined!")


apply_apb function defined!


In [4]:
# ============================================================================
# CELL 4: Merge and Prune Functions
# ============================================================================
def l1inftyinfty(filter: torch.Tensor) -> float:
    abs_w = filter.abs()
    max_w = abs_w.max(dim=2).values
    max_hw = max_w.max(dim=1).values
    return max_hw.sum().item()

def l1inftyinfty_distance(filter1: torch.Tensor, filter2: torch.Tensor) -> float:
    diff = torch.abs(filter1 - filter2)
    return l1inftyinfty(diff)

def get_weight(layer: nn.Module):
    if isinstance(layer, APBLayer):
        return layer.get_effective_weight()
    elif isinstance(layer, (nn.Conv2d, nn.Linear)):
        return layer.weight
    else:
        raise ValueError("Layer phải là APBLayer, nn.Conv2d hoặc nn.Linear.")

def compute_distance_matrix(layer: nn.Module):
    if not isinstance(layer.wrapped_layer if isinstance(layer, APBLayer) else layer, nn.Conv2d):
        raise ValueError("compute_distance_matrix chỉ hỗ trợ Conv2d.")

    weight = get_weight(layer)
    out_channels = weight.shape[0]
    n = out_channels
    D = torch.zeros(n, n, device=weight.device)
    max_dist, min_dist, sum_dist, count = 0.0, float('inf'), 0.0, 0
    zero_dist_count = 0

    for i in range(n):
        for j in range(i + 1, n):
            filter1 = weight[i]
            filter2 = weight[j]
            dist = l1inftyinfty_distance(filter1, filter2)
            D[i, j] = dist
            D[j, i] = dist
            max_dist = max(max_dist, dist)
            min_dist = min(min_dist, dist)
            sum_dist += dist * 2
            count += 2
            if dist == 0:
                zero_dist_count += 1

    mean_dist = sum_dist / count if count > 0 else 0.0
    print(f"Distance matrix: min={min_dist:.4f}, max={max_dist:.4f}, mean={mean_dist:.4f}")
    print(f"Identical filter pairs: {zero_dist_count}")
    return D

def merge_two_filters(f1: torch.Tensor, f2: torch.Tensor, mode: str = 'arithmetic'):
    if f1.shape != f2.shape:
        raise ValueError("Hai filters phải có shape giống nhau.")

    if mode == 'arithmetic':
        return (f1 + f2) / 2
    elif mode == 'geometric':
        abs_prod = torch.sqrt(torch.abs(f1) * torch.abs(f2))
        sign = torch.sign(f1 + f2)
        return sign * abs_prod
    elif mode == 'harmonic':
        abs1 = torch.abs(f1)
        abs2 = torch.abs(f2)
        harm = 2 * abs1 * abs2 / (abs1 + abs2 + 1e-8)
        sign = torch.sign(f1 + f2)
        return sign * harm
    elif mode == 'quadratic':
        quad = torch.sqrt((f1**2 + f2**2) / 2)
        sign = torch.sign(f1 + f2)
        return sign * quad
    else:
        raise ValueError(f"Mode không hỗ trợ: {mode}")


def apply_merge_by_ratio(layer: nn.Module, ratio: float, mode: str = 'arithmetic'):
    """
    Merge các filter Conv2d (hỗ trợ cả APB và Standard) dựa trên tỉ lệ.
    *** PHIÊN BẢN CẬP NHẬT: Xử lý cả 2 loại layer và đăng ký 'survival_mask' ***
    """
    # Kiểm tra xem layer gốc (wrapped) hoặc layer (standard) có phải là Conv2d không
    base_layer = layer.wrapped_layer if isinstance(layer, APBLayer) else layer
    if not isinstance(base_layer, nn.Conv2d):
        raise ValueError("apply_merge_by_ratio chỉ hỗ trợ Conv2d.")

    # 1. Lấy weight hiệu dụng và tính ma trận khoảng cách
    # Hàm get_weight() của bạn đã đúng, nó xử lý cả 2 loại layer
    weight_effective = get_weight(layer) 
    D = compute_distance_matrix(layer) # Hàm compute_distance_matrix() của bạn cũng đã đúng
    n = weight_effective.shape[0]

    # 2. Xác định số lượng filter cần loại bỏ
    num_to_remove = int(round(n * ratio))
    
    print(f"Số lượng filters: {n}, Tỉ lệ prune: {ratio*100:.1f}%")
    print(f"Mục tiêu loại bỏ: {num_to_remove} filters")

    if num_to_remove == 0:
        print("Không có filter nào bị merge.")
        # Vẫn đăng ký mask "sống sót 100%"
        all_indices = set(range(n))
        surviving_indices_list = sorted(list(all_indices))
        survival_mask = torch.tensor(surviving_indices_list, dtype=torch.long, device=weight_effective.device)
        layer.register_buffer('survival_mask', survival_mask)
        print("-> Đã đăng ký 'survival_mask' với 100% filter.")
        return layer

    # 3. LẤY ĐÚNG TRỌNG SỐ ĐỂ CẬP NHẬT
    if isinstance(layer, APBLayer):
        print("Updating 'latent_weight' (APBLayer)")
        weight_param = layer.latent_weight.data
    elif isinstance(layer, nn.Conv2d): # Đây là logic đúng
        print("Updating 'weight' (Standard nn.Conv2d)")
        weight_param = layer.weight.data
    else:
        raise ValueError("Layer không được hỗ trợ (Conv2d).")

    # 4. Tìm k cặp filter tốt nhất
    pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            pairs.append((D[i, j].item(), i, j))
    
    pairs.sort()

    processed = set()
    pruned_indices = set() 
    merge_count = 0
    
    for dist, i, j in pairs:
        if merge_count >= num_to_remove:
            break
        
        if i in processed or j in processed:
            continue

        # 5. Thực hiện merge: merge j vào i
        merged = merge_two_filters(weight_param[i], weight_param[j], mode) 
        
        # Cập nhật weight gốc (latent_weight hoặc weight)
        weight_param[i] = merged
        weight_param[j] = torch.zeros_like(weight_param[j])
        
        processed.add(i) 
        processed.add(j) 
        pruned_indices.add(j)
        
        merge_count += 1
        
    # === TẠO VÀ LƯU SURVIVAL MASK ===
    all_indices = set(range(n))
    surviving_indices_list = sorted(list(all_indices - pruned_indices))
    survival_mask = torch.tensor(surviving_indices_list, dtype=torch.long, device=weight_param.device)
    layer.register_buffer('survival_mask', survival_mask)
    # =======================================================
        
    print(f"Đã merge {merge_count} cặp filter (loại bỏ {merge_count} filters)")
    print(f"-> Đã đăng ký 'survival_mask' với {len(survival_mask)} filter còn sống.")
    torch.cuda.empty_cache()
    return layer

In [5]:
# ============================================================================
# CELL 5: Evaluation Function
# ============================================================================
def evaluate(model, data_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(data_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

print("Evaluate function defined!")

Evaluate function defined!


In [6]:
# ============================================================================
# CELL 6: Main Training Script
# ============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model
model_name = 'resnet18'
print(f"\nLoading pre-trained {model_name}...")
model = models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(model.fc.in_features, 10)

# Apply APB
print("\nApplying APB...")
model_apb = apply_apb(model)
model_apb.to(device)
print("APB applied!")

# Data loaders
print("\nPreparing CIFAR-10 data...")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Training setup
optimizer = optim.Adam(model_apb.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
total_epochs = 10
params_frozen = False
save_path_before = '/kaggle/working/before_prune.pth'
save_path_final = '/kaggle/working/final_model.pth'
best_accuracy = 0.0

print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

Using device: cuda

Loading pre-trained resnet18...

Applying APB...
Skipping APB for: conv1
Applying APB to: layer1.0.conv1
Applying APB to: layer1.0.conv2
Applying APB to: layer1.1.conv1
Applying APB to: layer1.1.conv2
Skipping APB for: layer2.0.conv1
Applying APB to: layer2.0.conv2
Skipping APB for: layer2.0.downsample.0
Applying APB to: layer2.1.conv1
Applying APB to: layer2.1.conv2
Skipping APB for: layer3.0.conv1
Applying APB to: layer3.0.conv2
Skipping APB for: layer3.0.downsample.0
Applying APB to: layer3.1.conv1
Applying APB to: layer3.1.conv2
Skipping APB for: layer4.0.conv1
Applying APB to: layer4.0.conv2
Skipping APB for: layer4.0.downsample.0
Applying APB to: layer4.1.conv1
Applying APB to: layer4.1.conv2
Skipping APB for: fc
APB applied!

Preparing CIFAR-10 data...

STARTING TRAINING


In [7]:
# Hàm load và đánh giá model
def load_and_evaluate_model(save_path, model_class, test_loader, criterion, device):
    # Khởi tạo model mới với cùng kiến trúc
    model = model_class().to(device)  # Thay model_class bằng lớp thực tế của model_apb
    
    # Load state từ file
    state = torch.load(save_path, map_location=device)
    
    # Tái tạo các tham số cho APBLayer
    for name, module in model.named_modules():
        if isinstance(module, APBLayer):
            # Lấy các tham số đã lưu
            weight_shape = state[f"{name}.weight_shape"]
            fp_positions = state[f"{name}.fp_positions"]
            fp_values = state[f"{name}.fp_values"]
            packed_signs = state[f"{name}.packed_signs"]
            alpha = state[f"{name}.alpha"]
            
            # Giải nén dấu (signs)
            sign_bits = torch.from_numpy(np.unpackbits(packed_signs.cpu().numpy())).to(device)[:weight_shape.numel()]
            sign_bits = sign_bits.view(weight_shape).to(torch.float32)
            full_signs = sign_bits * 2 - 1  # Chuyển {0,1} thành {-1,1}
            
            # Tái tạo weight (latent_weight)
            latent_weight = torch.zeros(weight_shape, device=device)
            latent_weight[fp_positions[:, 0], fp_positions[:, 1]] = fp_values  # Gán full-precision values
            # Tái tạo phần binary
            binary_part = alpha * full_signs
            latent_weight = latent_weight + binary_part  # Kết hợp full-precision và binary
            
            # Gán các tham số vào module
            module.latent_weight = nn.Parameter(latent_weight)
            module.alpha = nn.Parameter(alpha)
            # Gán delta mặc định vì delta đã bị xóa
            module.delta = nn.Parameter(torch.tensor(1e-8, device=device))  # Giá trị mặc định nhỏ
            
    # Load state vào model
    model.load_state_dict(state, strict=False)
    model.eval()
    
    # Đánh giá model
    test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)
    print(f"Loaded Model - Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")
    
    return test_loss, test_accuracy

In [8]:
# ============================================================================
# CELL 7: Training Loop
# ============================================================================
for epoch in range(total_epochs):
    if epoch >= total_epochs // 2 and not params_frozen:
        print("\n" + "="*40)
        print(f"EPOCH {epoch}: Freezing alpha and delta")
        print("="*40)
        for module in model_apb.modules():
            if isinstance(module, APBLayer):
                module.alpha.requires_grad = False
                module.delta.requires_grad = False
        params_frozen = True

    model_apb.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_apb(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    test_loss, test_accuracy = evaluate(model_apb, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{total_epochs}] - "
          f"Train Loss: {train_loss:.4f}, "
          f"Test Loss: {test_loss:.4f}, "
          f"Test Acc: {test_accuracy:.2f}%")

    # Save best model (FIXED: use numpy.packbits)
    if test_accuracy > best_accuracy:
        print(f"→ New best! Saving to {save_path_before}")
        best_accuracy = test_accuracy
        state = model_apb.state_dict()
        
        for name, module in model_apb.named_modules():
            if isinstance(module, APBLayer):
                weight = module.get_effective_weight()
                threshold = module.alpha.abs() + module.delta.clamp(min=1e-8)
                fp_mask = weight.abs() > threshold
                fp_positions = torch.nonzero(fp_mask, as_tuple=False)
                fp_values = weight[fp_mask]
                full_signs = torch.sign(weight)
                sign_bits = (full_signs > 0).to(torch.uint8).view(-1)
                
                # FIX: Use numpy.packbits instead of torch.packbits
                sign_bits_cpu = sign_bits.cpu().numpy()
                packed_signs = torch.from_numpy(np.packbits(sign_bits_cpu))
                
                state[f"{name}.alpha"] = module.alpha
                state[f"{name}.packed_signs"] = packed_signs
                state[f"{name}.weight_shape"] = torch.tensor(weight.shape)
                state[f"{name}.fp_positions"] = fp_positions
                state[f"{name}.fp_values"] = fp_values
                
                if f"{name}.latent_weight" in state:
                    del state[f"{name}.latent_weight"]
                if f"{name}.delta" in state:
                    del state[f"{name}.delta"]
        
        torch.save(state, save_path_before)
        print("Save successfully")

    # Print stats every 5 epochs
    if (epoch + 1) % 5 == 0:
        try:
            example_layer = model_apb.layer1[0].conv1
            if isinstance(example_layer, APBLayer):
                stats = example_layer.get_stats()
                print(f"  Stats (layer1.0.conv1): "
                      f"α={stats['alpha']:.4f}, δ={stats['delta']:.4f}, "
                      f"Binary={stats['percent_binary']:.2f}%")
        except:
            pass

print("\n" + "="*60)
print("TRAINING COMPLETE")
print(f"Best accuracy before pruning: {best_accuracy:.2f}%")
print("="*60)

Epoch [1/10] - Train Loss: 0.4783, Test Loss: 0.2703, Test Acc: 91.19%
→ New best! Saving to /kaggle/working/before_prune.pth
Save successfully
Epoch [2/10] - Train Loss: 0.2036, Test Loss: 0.2242, Test Acc: 92.55%
→ New best! Saving to /kaggle/working/before_prune.pth
Save successfully
Epoch [3/10] - Train Loss: 0.1267, Test Loss: 0.2399, Test Acc: 92.18%
Epoch [4/10] - Train Loss: 0.0872, Test Loss: 0.2437, Test Acc: 92.31%
Epoch [5/10] - Train Loss: 0.0691, Test Loss: 0.2612, Test Acc: 92.29%
  Stats (layer1.0.conv1): α=0.0316, δ=0.1602, Binary=98.75%

EPOCH 5: Freezing alpha and delta
Epoch [6/10] - Train Loss: 0.0552, Test Loss: 0.2465, Test Acc: 92.69%
→ New best! Saving to /kaggle/working/before_prune.pth
Save successfully
Epoch [7/10] - Train Loss: 0.0514, Test Loss: 0.2714, Test Acc: 92.33%
Epoch [8/10] - Train Loss: 0.0432, Test Loss: 0.2555, Test Acc: 92.47%
Epoch [9/10] - Train Loss: 0.0414, Test Loss: 0.2622, Test Acc: 92.84%
→ New best! Saving to /kaggle/working/before_pr

In [9]:
# ============================================================================
# CELL 8: Merge & Prune (ALL CONV LAYERS, SKIP layer1.0.conv1 and fc)
# ============================================================================
pruning_ratio = 0.4 
mode = 'harmonic' # Giữ nguyên 'harmonic' mode của bạn
print(f"\nApplying merge (ratio={pruning_ratio*100}%, mode={mode}) on ALL Conv layers (skipping Linear)...")

# Dùng set để tránh xử lý một layer 2 lần
processed_layers = set()

# Tên layer đặc biệt cần bỏ qua
SKIP_LAYER_NAME = 'layer1.0.conv1'

for name, module in model_apb.named_modules():
    if name in processed_layers:
        continue

    # Bỏ qua các wrapped_layer bên trong (để tránh lỗi AttributeError)
    if 'wrapped_layer' in name.split('.'):
        continue

    # === YÊU CẦU MỚI: Bỏ qua layer cụ thể (layer1.0.conv1) ===
    if name == SKIP_LAYER_NAME:
        print(f"\n--- SKIPPING {name} (as requested) ---")
        processed_layers.add(name)
        continue
    # =========================================================
        
    # --- Bỏ qua (SKIP) các lớp Linear (fc) ---
    if isinstance(module, nn.Linear):
        print(f"\n--- SKIPPING Linear layer: {name} (as requested) ---")
        processed_layers.add(name)
        continue

    # --- Ưu tiên 1: Xử lý các lớp APBLayer (Conv2d) ---
    if isinstance(module, APBLayer):
        if isinstance(module.wrapped_layer, nn.Conv2d):
            print(f"\n*** Merging APB-CONV2D layer: {name}")
            apply_merge_by_ratio(module, ratio=pruning_ratio, mode=mode)
            processed_layers.add(name)
        
    # --- Ưu tiên 2: Xử lý các lớp Standard Conv2d (conv1, downsample) ---
    elif isinstance(module, nn.Conv2d):
        print(f"\n*** Merging Standard-CONV2D layer: {name}")
        apply_merge_by_ratio(module, ratio=pruning_ratio, mode=mode)
        processed_layers.add(name)


# Evaluate after merge
test_loss, test_accuracy = evaluate(model_apb, test_loader, criterion, device)
print(f"\nPost-merge - Loss: {test_loss:.4f}, Acc: {test_accuracy:.2f}%")

# Reset best_accuracy để chuẩn bị cho CELL 9 (Fine-tuning)
print(f"Resetting best accuracy from {best_accuracy:.2f}% to {test_accuracy:.2f}% for fine-tuning.")
best_accuracy = test_accuracy


Applying merge (ratio=40.0%, mode=harmonic) on ALL Conv layers (skipping Linear)...

*** Merging Standard-CONV2D layer: conv1
Distance matrix: min=0.0000, max=4.2369, mean=1.4795
Identical filter pairs: 0
Số lượng filters: 64, Tỉ lệ prune: 40.0%
Mục tiêu loại bỏ: 26 filters
Updating 'weight' (Standard nn.Conv2d)
Đã merge 26 cặp filter (loại bỏ 26 filters)
-> Đã đăng ký 'survival_mask' với 38 filter còn sống.

--- SKIPPING layer1.0.conv1 (as requested) ---

*** Merging APB-CONV2D layer: layer1.0.conv2
Distance matrix: min=3.8223, max=7.1026, mean=5.1088
Identical filter pairs: 0
Số lượng filters: 64, Tỉ lệ prune: 40.0%
Mục tiêu loại bỏ: 26 filters
Updating 'latent_weight' (APBLayer)
Đã merge 26 cặp filter (loại bỏ 26 filters)
-> Đã đăng ký 'survival_mask' với 38 filter còn sống.

*** Merging APB-CONV2D layer: layer1.1.conv1
Distance matrix: min=4.1805, max=8.4904, mean=5.6235
Identical filter pairs: 0
Số lượng filters: 64, Tỉ lệ prune: 40.0%
Mục tiêu loại bỏ: 26 filters
Updating 'laten

In [10]:
# ============================================================================
# CELL 9: Fine-tuning After Merge (LOGIC LƯU FILE THÔNG MINH)
# ============================================================================
fine_tune_epochs = 50
print(f"\nFine-tuning for {fine_tune_epochs} epochs...")

for epoch in range(fine_tune_epochs):
    model_apb.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_apb(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    test_loss, test_accuracy = evaluate(model_apb, test_loader, criterion, device)
    print(f"FT Epoch [{epoch+1}/{fine_tune_epochs}] - "
          f"Train Loss: {train_loss:.4f}, "
          f"Test Loss: {test_loss:.4f}, "
          f"Test Acc: {test_accuracy:.2f}%")

    if test_accuracy > best_accuracy:
        print(f"→ New best! Saving to {save_path_final}")
        best_accuracy = test_accuracy
        
        # Lấy state_dict. Giờ nó sẽ chứa 'survival_mask'
        # cho các layer đã được prune ở CELL 8.
        state_after_merge = model_apb.state_dict()
        
        processed_layers_save = set()

        for name, module in model_apb.named_modules():
            if name in processed_layers_save or 'wrapped_layer' in name.split('.'):
                continue

            # --- 1. XỬ LÝ CÁC LỚP APBLayer (vd: layer1.0.conv2) ---
            if isinstance(module, APBLayer):
                processed_layers_save.add(name)
                
                # KIỂM TRA XEM LAYER NÀY CÓ BỊ PRUNE CẤU TRÚC KHÔNG
                if f"{name}.survival_mask" in state_after_merge:
                    # ---- LOGIC LƯU THƯA (SPARSE APB) ----
                    print(f"  Saving {name} (Structurally Sparse APB)...")
                    
                    weight = module.get_effective_weight()
                    survival_mask = state_after_merge[f"{name}.survival_mask"]
                    # Chỉ lấy các trọng số CÒN SỐNG
                    surviving_weights = weight[survival_mask]
                    
                    threshold = module.alpha.abs() + module.delta.clamp(min=1e-8)
                    
                    fp_mask = surviving_weights.abs() > threshold
                    fp_positions = torch.nonzero(fp_mask, as_tuple=False) 
                    fp_values = surviving_weights[fp_mask]
                    
                    full_signs = torch.sign(surviving_weights)
                    sign_bits = (full_signs > 0).to(torch.uint8).view(-1)
                    packed_signs = torch.from_numpy(np.packbits(sign_bits.cpu().numpy()))

                    # Ghi đè các key trong state_dict
                    state_after_merge[f"{name}.alpha"] = module.alpha
                    state_after_merge[f"{name}.original_shape"] = torch.tensor(weight.shape) 
                    state_after_merge[f"{name}.packed_signs"] = packed_signs 
                    state_after_merge[f"{name}.fp_positions"] = fp_positions 
                    state_after_merge[f"{name}.fp_values"] = fp_values 
                    
                    if f"{name}.weight_shape" in state_after_merge:
                        del state_after_merge[f"{name}.weight_shape"]

                else:
                    # ---- LOGIC LƯU DÀY (DENSE APB) ----
                    # Dành cho các lớp APB không bị prune (vd: layer1.0.conv1)
                    print(f"  Saving {name} (Dense APB - Skipped)...")
                    weight = module.get_effective_weight()
                    threshold = module.alpha.abs() + module.delta.clamp(min=1e-8)
                    fp_mask = weight.abs() > threshold
                    fp_positions = torch.nonzero(fp_mask, as_tuple=False)
                    fp_values = weight[fp_mask]
                    full_signs = torch.sign(weight)
                    sign_bits = (full_signs > 0).to(torch.uint8).view(-1)
                    
                    sign_bits_cpu = sign_bits.cpu().numpy()
                    packed_signs = torch.from_numpy(np.packbits(sign_bits_cpu))
                    
                    state_after_merge[f"{name}.alpha"] = module.alpha
                    state_after_merge[f"{name}.packed_signs"] = packed_signs
                    state_after_merge[f"{name}.weight_shape"] = torch.tensor(weight.shape)
                    state_after_merge[f"{name}.fp_positions"] = fp_positions
                    state_after_merge[f"{name}.fp_values"] = fp_values

                # Xóa các tham số huấn luyện (áp dụng cho cả 2 trường hợp)
                if f"{name}.latent_weight" in state_after_merge:
                    del state_after_merge[f"{name}.latent_weight"]
                if f"{name}.delta" in state_after_merge:
                    del state_after_merge[f"{name}.delta"]

            # --- 2. XỬ LÝ CÁC LỚP STANDARD CONV2D (vd: conv1, downsample) ---
            elif isinstance(module, nn.Conv2d):
                processed_layers_save.add(name)
                
                # Chỉ xử lý nếu nó có survival_mask (tức là CELL 8 đã prune nó)
                if f"{name}.survival_mask" in state_after_merge:
                    print(f"  Saving {name} (Structurally Sparse Standard Conv2d)...")
                    
                    weight = module.weight.data # Lấy weight standard
                    survival_mask = state_after_merge[f"{name}.survival_mask"]
                    
                    # Chỉ lưu các filter còn sống
                    surviving_weights = weight[survival_mask] 
                    
                    # Cập nhật state_dict để chỉ lưu các trọng số đã prune
                    state_after_merge[f"{name}.weight"] = surviving_weights
                    # Lưu shape gốc để biết cách load lại
                    state_after_merge[f"{name}.original_shape"] = torch.tensor(weight.shape)
                    # survival_mask đã có sẵn trong state_dict
                else:
                    # Lớp Conv2d không bị prune (ví dụ: không có)
                    pass

            # --- 3. XỬ LÝ CÁC LỚP STANDARD LINEAR (fc) ---
            elif isinstance(module, nn.Linear):
                processed_layers_save.add(name)
                # Bỏ qua, không làm gì cả (vì CELL 8 cũng bỏ qua)
                print(f"  Saving {name} (Standard Linear - Skipped)...")
                pass

        torch.save(state_after_merge, save_path_final)
        print("Save successfully (structurally aware)")

print("\n" + "="*60)
print("ALL TRAINING COMPLETE!")
print(f"Final best accuracy: {best_accuracy:.2f}%")
print(f"Models saved to:")
print(f"  - {save_path_before}")
print(f"  - {save_path_final}")
print("="*60)


Fine-tuning for 50 epochs...
FT Epoch [1/50] - Train Loss: 0.9253, Test Loss: 0.4916, Test Acc: 83.06%
→ New best! Saving to /kaggle/working/final_model.pth
  Saving conv1 (Structurally Sparse Standard Conv2d)...
  Saving layer1.0.conv1 (Dense APB - Skipped)...
  Saving layer1.0.conv2 (Structurally Sparse APB)...
  Saving layer1.1.conv1 (Structurally Sparse APB)...
  Saving layer1.1.conv2 (Structurally Sparse APB)...
  Saving layer2.0.conv1 (Structurally Sparse Standard Conv2d)...
  Saving layer2.0.conv2 (Structurally Sparse APB)...
  Saving layer2.0.downsample.0 (Structurally Sparse Standard Conv2d)...
  Saving layer2.1.conv1 (Structurally Sparse APB)...
  Saving layer2.1.conv2 (Structurally Sparse APB)...
  Saving layer3.0.conv1 (Structurally Sparse Standard Conv2d)...
  Saving layer3.0.conv2 (Structurally Sparse APB)...
  Saving layer3.0.downsample.0 (Structurally Sparse Standard Conv2d)...
  Saving layer3.1.conv1 (Structurally Sparse APB)...
  Saving layer3.1.conv2 (Structurally S